# Lesson 1.2 — 动作空间（action space）与控制模式（control mode）

一个 action 永远无法仅由其 tensor 完整描述。本 notebook 通过**同一个 task** 上改变一个参数 —— `control_mode` —— 并观察底层的 action space 如何随之变化，把这一点具体化。

本课的 Action Specification 为：

```text
(space, representation, frame, dimension, semantics, unit, frequency, controller mode)
```

下面对比表的每一行都填充了该 tuple 的一个不同子集。
维度（dimension）匹配并**不**意味着 semantics 匹配。


## 1.2.1 — 共享 preamble


In [1]:
import gymnasium as gym
import mani_skill.envs
import numpy as np
import torch

torch.set_printoptions(precision=4, sci_mode=False)


def make_env(obs_mode="state", control_mode="pd_joint_delta_pos", seed=0):
    env = gym.make(
        "PickCube-v1",
        obs_mode=obs_mode,
        control_mode=control_mode,
        num_envs=1,
    )
    env.reset(seed=seed)
    return env

## 1.2.2 — 枚举所有 control mode

同一个 `PickCube-v1` task、十一种 control mode，以及从 **4 到 15** 不等的
action 维度（dimension）。认识到这一取值范围才是重点：“一个 8 维 action”本身说明不了任何问题。


In [2]:
agent = make_env().unwrapped.agent
modes = sorted(agent.supported_control_modes)

print(f"{'control mode':<28} {'action space':<52} dim")
print("-" * 92)
for mode in modes:
    try:
        e = make_env(control_mode=mode)
        space = e.action_space
        shape = space.shape
        if np.allclose(space.low, -1.0) and np.allclose(space.high, 1.0):
            desc = f"Box(-1, 1, {shape})  normalized"
        else:
            desc = f"Box(lo, hi, {shape})  physical limits"
        print(f"{mode:<28} {desc:<52} {shape[0]}")
        e.close()
    except Exception as exc:
        print(f"{mode:<28} FAILED: {type(exc).__name__}: {exc}")

2026-09-22 11:14:12,657 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1113: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 304: OS call failed or operation not supported on this OS (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


2026-09-22 11:14:13,012 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


2026-09-22 11:14:13,159 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


control mode                 action space                                         dim
--------------------------------------------------------------------------------------------
pd_ee_delta_pos              Box(-1, 1, (4,))  normalized                         4


...

2026-09-22 11:14:14,184 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


2026-09-22 11:14:14,321 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


pd_joint_pos                 Box(lo, hi, (8,))  physical limits                   8
pd_joint_pos_vel             Box(lo, hi, (15,))  physical limits                  15


2026-09-22 11:14:14,451 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


pd_joint_target_delta_pos    Box(-1, 1, (8,))  normalized                         8
pd_joint_vel                 Box(-1, 1, (8,))  normalized                         8


### 为什么维度不同

| Control mode | 维度 | Semantics |
|---|---:|---|
| `pd_joint_delta_pos` | 8 | 7 个 joint delta + 1 个 gripper |
| `pd_joint_pos` | 8 | 7 个 absolute joint target + 1 个 gripper，**physical limits** 而非 `[-1,1]` |
| `pd_joint_vel` | 8 | 7 个 joint velocity + gripper |
| `pd_joint_pos_vel` | 15 | 8 个 position + 7 个 velocity |
| `pd_ee_delta_pos` | 4 | 3 个 Cartesian delta + gripper |
| `pd_ee_delta_pose` | 7 | 6-DoF Cartesian delta + gripper |
| `pd_ee_pose` | 7 | absolute end-effector pose target |

由此表可得出两个事实：

- **joint space 与 Cartesian space**：`pd_joint_*` 以 joint angle 为目标，
  `pd_ee_*` 以 end-effector 为目标。同一物理运动在两者中是不同的 action。
- **delta 与 absolute**：`pd_joint_delta_pos` 与 `pd_joint_pos` 的维度都是 8，
  但一个是 displacement，另一个是 position。这是对“shape 相同即 action 相同”
  最清楚的反例。


## 1.2.3 — 本项目的 control mode：`pd_joint_delta_pos`

本仓库中的 dataset 是用 `pd_joint_delta_pos` 采集的。它的 8 个通道是
**两种不同 semantics 的拼接**：

| 通道 | Controller | Semantics | 物理范围 |
|---|---|---|---|
| `a[0:7]` | `PDJointPosController` | 相对当前 joint position 的 normalized **delta** | `[-0.1, 0.1]` rad |
| `a[7]` | `PDJointPosMimicController` | normalized **absolute** gripper target | `[-0.01, 0.04]` m |

应当直接从运行中的 controller 读取 config，而不是相信命名。


In [3]:
env = make_env()
controller = env.unwrapped.agent.controller

print("action_mapping:", controller.action_mapping)
print("action space   :", controller.action_space)

for name, config in controller.configs.items():
    print(f"\n--- {name} ({type(config).__name__}) ---")
    for field in ("joint_names", "lower", "upper", "use_delta", "use_target",
                  "normalize_action", "mimic", "stiffness", "damping"):
        print(f"    {field:<18} {getattr(config, field, '<absent>')}")

2026-09-22 11:14:14,583 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


action_mapping: {'arm': (0, 7), 'gripper': (7, 8)}
action space   : Box(-1.0, 1.0, (8,), float32)

--- arm (PDJointPosControllerConfig) ---
    joint_names        ['panda_joint1', 'panda_joint2', 'panda_joint3', 'panda_joint4', 'panda_joint5', 'panda_joint6', 'panda_joint7']
    lower              -0.1
    upper              0.1
    use_delta          True
    use_target         False
    normalize_action   True
    mimic              <absent>
    stiffness          1000.0
    damping            100.0

--- gripper (PDJointPosMimicControllerConfig) ---
    joint_names        ['panda_finger_joint1', 'panda_finger_joint2']
    lower              -0.01
    upper              0.04
    use_delta          False
    use_target         False
    normalize_action   True
    mimic              {'panda_finger_joint2': {'joint': 'panda_finger_joint1'}}
    stiffness          1000.0
    damping            100.0


请仔细阅读这些 flag，因为它们共同决定了一个 action
*意味着*什么：

- 机械臂上的 `use_delta=True` —— action 是一个 displacement。
- `use_target=False` —— delta 是相对**当前** joint
  position 度量的，而不是相对上一次下发的 target。
- `normalize_action=True` —— `[-1,1]` 的输入会被缩放到 `[lower, upper]`。
- 对 gripper 而言，`use_delta=False` —— 它是 absolute position target，尽管
  它位于同一个 normalized `[-1,1]` 区间内。


## 1.2.4 — 用实验验证机械臂的映射关系

有两个数字很重要，而且它们并不相同：

1. 一步之后的 **control target** —— 它应当恰好移动 `0.1 · a`；
2. 一步之后的 **actual joint position** —— 它移动得少得多，因为
   PD controller 需要经过若干步才跟踪上 target。

把两者混淆，很容易误读 action 的幅值。


In [4]:
env = make_env()
robot = env.unwrapped.agent.robot


def finger_indices(robot):
    names = [joint.name for joint in robot.active_joints]
    return names.index("panda_finger_joint1"), names.index("panda_finger_joint2")


for value in (1.0, -1.0):
    env.reset(seed=0)
    qpos_before = robot.get_qpos().clone()[0]

    action = torch.zeros(1, 8)
    action[0, :7] = value
    env.step(action)

    target = robot.get_drive_targets()
    target = target[0] if target.ndim > 1 else target

    print(f"a_arm = {value:+.1f}")
    print(f"   command implied delta : {0.1 * value:+.4f} rad")
    print(f"   drive_target - qpos   : {target[:3].cpu().numpy()}  <-- exact")
    print(f"   qpos_after - qpos     : {(robot.get_qpos()[0] - qpos_before)[:3].cpu().numpy()}  <-- partial (PD tracking)")
    print()

2026-09-22 11:14:14,661 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


a_arm = +1.0
   command implied delta : +0.1000 rad
   drive_target - qpos   : [0.13528106 0.50070226 0.11957476]  <-- exact
   qpos_after - qpos     : [0.0131324  0.03490701 0.02570832]  <-- partial (PD tracking)

a_arm = -1.0
   command implied delta : -0.1000 rad
   drive_target - qpos   : [-0.06471895  0.30070224 -0.08042524]  <-- exact
   qpos_after - qpos     : [-0.01323517 -0.03481296 -0.02594179]  <-- partial (PD tracking)



## 1.2.5 — 验证 gripper 的映射关系与方向

gripper 是 absolute 的，因此对同一个 action，它的 target 应当是固定值，
与当前 state 无关。而方向问题 —— `+1` 是张开还是闭合？—— 可以通过测量来回答，
而不是靠符号去猜。


In [5]:
env = make_env()
robot = env.unwrapped.agent.robot
f1, f2 = finger_indices(robot)

for value in (-1.0, 0.0, 1.0):
    env.reset(seed=0)
    qpos_before = robot.get_qpos().clone()[0]

    action = torch.zeros(1, 8)
    action[0, 7] = value
    env.step(action)

    target = robot.get_drive_targets()
    target = target[0] if target.ndim > 1 else target

    print(f"a_gripper = {value:+.1f} -> target finger1 = {target[f1].item():+.5f}, "
          f"finger2 = {target[f2].item():+.5f}   (reset qpos was {qpos_before[f1].item():+.4f})")

print("\nBoth fingers receive the same target because finger2 mimics finger1.")
print("The gripper joint starts at its maximum value, so +1 is the open extreme.")

2026-09-22 11:14:14,758 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


a_gripper = -1.0 -> target finger1 = -0.01000, finger2 = -0.01000   (reset qpos was +0.0400)
a_gripper = +0.0 -> target finger1 = +0.01500, finger2 = +0.01500   (reset qpos was +0.0400)


a_gripper = +1.0 -> target finger1 = +0.04000, finger2 = +0.04000   (reset qpos was +0.0400)

Both fingers receive the same target because finger2 mimics finger1.
The gripper joint starts at its maximum value, so +1 is the open extreme.


## 1.2.6 — Action Specification 表

为项目 dataset 填写此表。在比较两个 dataset 之前，每个字段都是必需的：

| 字段 | 值 |
|---|---|
| space | `Box(-1, 1, (8,), float32)` |
| representation | joint-space，混合 delta/absolute |
| coordinate frame | joint space（通道 0–6 不涉及 Cartesian frame） |
| dimension | 8 = 7 arm + 1 gripper |
| semantics | arm：joint position **delta**；gripper：**absolute** position target |
| unit | rad（arm），m（gripper joint position） |
| frequency | 20 Hz control（见 `2.3_time_alignment.ipynb`） |
| controller mode | `pd_joint_delta_pos` |

注意最后一行。本 notebook 确定的是 semantics；而 **frequency** 是一个
独立的属性，目前它在 dataset metadata 与环境之间并不一致。这正是
time-alignment notebook 的主题。


## 小结

1. task 的一个参数就把 action 维度从 4 变到 15。仅凭 shape 永远不构成一个
   specification。
2. `pd_joint_delta_pos` 在**同一个 vector 中具有两种 semantics**：机械臂的
   joint delta，以及 gripper 的 absolute target。
3. `use_target=False` 意味着 delta 施加在当前 joint position 上，因此
   同一个 action 从不同的 state 出发并不会产生相同的 absolute 运动。
4. 由于 PD controller 会随时间跟踪它的 target，在一个 step 内下发的 delta
   与实际实现的运动并不相同。
5. 两个 action shape 完全相同的 dataset 仍可能互不兼容 —— 例如
   `pd_joint_delta_pos` 与 `pd_joint_pos`，两者的 shape 都是 `(8,)`。
